In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

api = wandb.Api()

# Single parameter test
param = "beta_ee_3"
prefixed_param = f"best_sim_params/{param}"

project_path = "entity_name/sim_param_logging_v24"  # Replace with correct entity/project
runs = api.runs(project_path)

data = []
for run in runs:
    seed = run.config.get("random_seed")
    transitions = run.config.get("num_offline_collected_transitions")

    # Fetch only _step and this single parameter
    history_df = run.history(keys=["_step", "best_sim_params/beta_ee_3"], pandas=True)
    
    # Print columns for debugging
    print(f"Run: {run.name}")
    print("History columns:", history_df.columns.tolist())
    
    if prefixed_param not in history_df.columns:
        print(f"Run {run.name} does not have {prefixed_param}. Skipping.")
        continue
    
    # Drop rows where the parameter is NaN
    filtered_df = history_df.dropna(subset=[prefixed_param])
    if filtered_df.empty:
        print(f"Run {run.name} never logged a non-null value for {prefixed_param}. Skipping.")
        continue
    
    # Take the last step where the parameter was logged
    final_values = filtered_df.iloc[-1]
    value = final_values[prefixed_param]
    
    row = {
        "random_seed": seed,
        "num_offline_collected_transitions": transitions,
        param: value
    }
    data.append(row)

if not data:
    raise ValueError(f"No runs had the parameter {prefixed_param} logged. Check your logging code.")

df = pd.DataFrame(data)
df = df.sort_values(by="num_offline_collected_transitions")

# Plot the single parameter
plt.figure(figsize=(8, 6))
sns.lineplot(
    data=df,
    x="num_offline_collected_transitions",
    y=param,
    hue="random_seed",
    marker="o"
)
plt.title(param)
plt.xlabel("num_offline_collected_transitions")
plt.ylabel("Value")
plt.show()

wandb: ERROR keys must be specified in a list


AttributeError: 'list' object has no attribute 'columns'